# Chapter 1

- Data warehouse : Serves as a single source of truth that contains cleansed, conformed, and categorized data from multiple sources for different analytical purposes. 
- some providers are: Amazon Redshift, Google Bigquery, Snowflake, Cloudera

<center><img src="images/01.01.png"  style="width: 400px, height: 300px;"/></center>
<center><img src="images/01.02.png"  style="width: 400px, height: 300px;"/></center>

### Snowflake Architecture

- Decoupling Storage & Compute:
    - Efficient data storage.
    - Independent data processing.
    - Components operate without interdependence.
- Benefits
    - Enhanced scalability.
    - Faster data processing and response.
    - Cost-effective operations.
- Storage Layer: Columnar storage, Optimized analysis and organization by default, Compressed, Structured (Tables, schemas, databases)
- Compute Layer: Query execution, Virtual warehouses (temporary computing resources that handle different part of query and perform different operations like different experts working together to solve different problems), Scalability, Performance, Cost-effectiveness
- Cloud Services Layer (Acts as an Admin): Infrastructure management, Query Optimization, Authentication, Access control, Security

<center><img src="images/01.03.png"  style="width: 400px, height: 300px;"/></center>
<center><img src="images/01.04.png"  style="width: 400px, height: 300px;"/></center>


### Snowflake vs Competitors

<center><img src="images/01.05.png"  style="width: 400px, height: 300px;"/></center>
<center><img src="images/01.06.png"  style="width: 400px, height: 300px;"/></center>


# Chapter 2

- Raw data sources: Initial, unprocessed data, e.g., CSV files
- Staging: temporary location for storing data
    - Snowflake Staging is an intermediary storage area used for storing data files before they are loaded into Snowflake tables.
    - Internal Stage: When data is stored in Snowflake
    - External Stage: When data is stored in Amazon S3, Google Cloud Storage
- Table: Final data is loaded here


<center><img src="images/02.01.png"  style="width: 400px, height: 300px;"/></center>


### Snowflake Commands

```
-- Create a stage and put a file in that staging area
CREATE STAGE my_local_stage -- create a stage
PUT file:///path_to_your_local_file/filename.csv 
@my_local_stage -- put a file in the created stage 

-- Create a table
CREATE TABLE table_name (
col_id NUMBER COMMENT 'This is a Unique identifier',
col_date DATE,
col_time TIME,
col_datetime TIMESTAMP
)
COMMENT = 'This is a table'

-- load file from staging area to a table
COPY INTO table_name FROM @my_local_stage/filename.csv
FILE_FORMAT = (TYPE = 'CSV' SKIP_HEADER=1 )

-- Show information
SHOW DATABASES -- show all databases
SHOW [TABLES|SCHEMAS|VIEWS] IN DATABASE db_name -- show tables or schemas in a database
SHOW COLUMNS IN table_name -- show columns in a table
SHOW TABLES LIKE '%PIZZA%' IN DATABASE db_name -- show all tables with name that has word "PIZZA" in it

-- See description
DESCRIBE [DATABASE|TABLE|VIEW|STAGE] name
DESCRIBE SCHEMA PUBLIC -- see public schema

-- Create a view
CREATE VIEW view_name AS
SELECT col1, col2
FROM table_name

-- Renaming
ALTER TABLE IF EXISTS old_table_name RENAME TO new_table_name;
ALTER TABLE IF EXISTS table_name RENAME COLUMN old_col_name TO new_col_name;

-- Dropping a table
DROP TABLE table_name

-- INSERT using specified values
INSERT INTO table_name (id, date, time) VALUES (1, '2015-01-01', '11:38:36')
-- INSERT using query
INSERT INTO table_name
    SELECT * FROM some_table WHERE date > '2015-01-02'
-- UPDATE
UPDATE table_name
SET time = '17:00:00' WHERE id = '5'

-- UPSERT or synchronization update or insert
MERGE INTO target_table AS t
USING source_table AS s
ON t.id = s.id
WHEN MATCHED THEN UPDATE SET t.amount = s.amount
WHEN NOT MATCHED THEN INSERT (id, product, amount) VALUES (s.id, s.product, s.amount);

-- Top 10 rows
SELECT TOP 10*
FROM table_name
-- Casting
SELECT CAST(col_date AS TIMESTAMP) AS col_timestamp FROM orders
-- Conversion functions 
SELECT [TO_DATE|TO_VARCHAR|AVG|CONCAT|UPPER] (col_name) from table_name
-- standalone functions
SELECT [CURRENT_DATE|CURRENT_TIME]

-- extract date information
SELECT EXTRACT(YEAR FROM col_timestamp) AS year FROM table_name

-- group by operation
SELECT col1, col2, AVG(col3) AS average_val
FROM table_name
GROUP BY ALL -- No need to specify columns / you can specify col1, col2 if you want, which also works 
ORDER BY pizza_type_id, average_price DESC
-- JOIN
SELECT * FROM 
left_table AS l 
JOIN right_table AS r
ON r.id = l.id
-- NATURAL JOIN : Automatically joins on matching columns and does not return duplicate column values from both table (ON not allowed)
SELECT * FROM 
left_table AS l 
NATURAL JOIN right_table AS r

-- LATERAL JOIN : dynamic result based on subquery
SELECT *
FROM left_table AS outer, -- notice the comma here
LATERAL -- Keyword LATERAL for joining
( SELECT *
FROM right_table AS inner
WHERE outer.id = inner.id) AS lat

-- Snowflake does not allow LIMIT keyword on correlated subqueries, use CTE
-- CTE
WITH cte AS ( 
    SELECT id, MAX(price) AS max_price
    FROM some_table GROUP BY id )
SELECT l.col1, r.col2,
FROM left_table AS l
JOIN right_table AS r ON l.id = r.id
JOIN cte ON r.col3 = cte.id
WHERE l.price < cte.max_price

-- splot query history for analyzing query performance
SELECT *
FROM snowflake.account_usage.query_history
WHERE execution_time > 1000
```

### JSON 

```
-- parse string to json format
SELECT PARSE_JSON(json_col) AS customer_info_json
-- construct JSON object
SELECT OBJECT_CONSTRUCT(
key1,val1,
key2,val2
)

-- Separate the JSON data using colon to access data from column
SELECT
JSON_col:field1, 
JSON_col:field2,
JSON_col:field3,
FROM
talbe_name;
-- Separate the nested JSON data using consecutive colon to access data from column
SELECT
customer_info:address:street AS street_name
FROM cust_json_table

```

### Best Practices

- Do not use * in SELECT
- NEVER forget to use ON in a JOIN (Or it will be cross join exhaustion)
- Use UNION ALL over UNION if possible
- Apply LIMIT in CTE before joining with other table
- Use WHERE before JOIN to perform join on limited number of rows (Early filtering)
